# Use the generic offsettings models

This tutorial shows how to use the generic offsettings module of AeroMAPS. Similarly to the generic energy module, the offsetting mechanisms are defined by the user in a dedicated yaml configuration file. This makes it possible to decompose the total carbon offsetting demand of the aviation sector between different mechanisms, and in particular to distinguish **carbon dioxide removal** (CDR, e.g. DACCS or BECCS) from **other offsetting mechanisms** such as emissions avoidance credits.

The module works as follows:

1. The total carbon offsetting demand (`carbon_offset` variable) is computed upstream by the level and residual carbon offset models (`models_offset` standard models).
2. Each offsetting mechanism defined in the yaml file takes a part of this demand, either as a **share** of the total demand (`usage_type: share`) or directly as a **quantity** of CO2 (`usage_type: quantity`). The mechanism flagged as `default: True` fulfills the remaining demand.
3. A top-down cost model computes the cost of each mechanism from its unit cost trajectory [€/tCO2], possibly corrected by subsidies and taxes.
4. Aggregates are computed for each mechanism **category** (e.g. `carbon_dioxide_removal`, `emissions_avoidance`): carbon offset, share, cumulative carbon offset and cost, as well as the total offsetting cost and the mean carbon offset price.

The module is activated by adding the `models.offsettings` key in the configuration file:

```yaml
models:
  offsettings:
    offsettings_model_data_file: "./offsettings_data.yaml"
```


## Load and process

First, the user has to load the framework and generate a process using a configuration file which activates the offsettings models.


In [ ]:
%matplotlib widget
from aeromaps import create_process

In [ ]:
process = create_process(configuration_file="data/config.yaml")

## Offsetting mechanisms configuration

In this tutorial, three offsetting mechanisms are defined in `data/offsettings_data.yaml`:

- `daccs`: Direct Air Carbon Capture and Storage, a carbon dioxide removal mechanism defined by a **quantity** trajectory [MtCO2], with a decreasing unit cost and an increasing subsidy;
- `beccs`: Bioenergy with Carbon Capture and Storage, a carbon dioxide removal mechanism defined by a **share** of the total offsetting demand [%];
- `avoidance_credits`: emissions avoidance credits, flagged as the **default** mechanism, which fulfill the offsetting demand not covered by the two other mechanisms.

The mechanisms metadata can be accessed through the offsettings manager of the process.


In [ ]:
for mechanism in process.offsettings_manager.get_all():
    print(mechanism)

## Set up variables

The user can then set the different parameters of the model to generate its scenario. Here, a carbon offsetting strategy is defined through the residual carbon offset share, which leads to a non-zero carbon offsetting demand.


In [ ]:
# Carbon offsetting demand
process.parameters.carbon_offset_baseline_level_vs_2019_reference_periods = [2020, 2024, 2050]
process.parameters.carbon_offset_baseline_level_vs_2019_reference_periods_values = [100.0, 85.0]
process.parameters.residual_carbon_offset_share_reference_years = [2020, 2030, 2040, 2050]
process.parameters.residual_carbon_offset_share_reference_years_values = [0.0, 10.0, 30.0, 50.0]

The inputs defined in the offsettings yaml file can also be modified directly as parameters. For instance, the BECCS share of the total offsetting demand can be updated as follows.


In [ ]:
# Update the BECCS share trajectory defined in the yaml file
process.parameters.beccs_usage_share_years = [2020, 2030, 2040, 2050]
process.parameters.beccs_usage_share_values = [0.0, 10.0, 20.0, 30.0]

## Compute

Once all the parameters have been set up, the user can compute.


In [ ]:
process.compute()

## Results

The carbon offset and costs of each mechanism and category are available in the vector outputs. For instance, the decomposition of the carbon offset between the mechanisms and the corresponding costs can be displayed.


In [ ]:
process.data["vector_outputs"][
    [
        "carbon_offset",
        "daccs_carbon_offset",
        "beccs_carbon_offset",
        "avoidance_credits_carbon_offset",
        "carbon_dioxide_removal_carbon_offset",
        "emissions_avoidance_carbon_offset",
    ]
].loc[2025:2050:5]

In [ ]:
process.data["vector_outputs"][
    [
        "carbon_offset_total_cost",
        "carbon_offset_mean_price",
        "daccs_carbon_offset_cost",
        "beccs_carbon_offset_cost",
        "avoidance_credits_carbon_offset_cost",
        "carbon_dioxide_removal_cumulative_carbon_offset",
    ]
].loc[2025:2050:5]

## Plots

Dedicated plots, based on the offsettings manager, are available to display the results of the offsettings models.


In [ ]:
process.plot("carbon_offset_mix", save=False)

In [ ]:
process.plot("carbon_offset_mechanisms_breakdown", save=False)

In [ ]:
process.plot("carbon_offset_cost_breakdown", save=False)

In [ ]:
process.plot("carbon_offset_prices", save=False)

In [ ]:
from aeromaps.utils.functions import clean_notebooks_on_tests

clean_notebooks_on_tests(globals())